<a href="https://colab.research.google.com/github/Alice-Ferri/data-intensive-project-2026/blob/main/data-intensive-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Determinare la categoria della malattia della cellula.

Programmazione di data intensive a.a. 2025/2026

Alice Ferri, alice.ferri8@studio.unibo.it

Davide rossi, davide.rossi47@studio.unibo.it

### Parte 1 - Descrizione del contesto del problema

Il dataset di riferimento è [rilevamento di anomalie di cellule del sangue](https://www.kaggle.com/datasets/alitaqishah/blood-cell-anomaly-detection-2025/data) presente in Kaggle.

Il dataset contiene le informazioni di cellule del sangue sane e malate.
Tali dati permettono di suddividere le cellule in 7 categorie, 5 di queste classificate come anomale e 2 come normali.

L'obbiettivo del progetto è sviluppare un classificatore che sia in grando di determinare la categoria della cellula.

### Caricamento librerie

Installiamo nel kernel la libreria per importare il dataset da kagglehub

In [14]:
pip install kagglehub[pandas-datasets]

Note: you may need to restart the kernel to use updated packages.


Importiamo le librerie che utilizzeremo

In [16]:
import kagglehub
import numpy as np
import pandas as pd
from kagglehub import KaggleDatasetAdapter


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\alicf\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\alicf\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alicf\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\alicf\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

### Caricamento dei dati e preprocessing

Carichiamo il dataset in un pandas dataframe da kagglehub

In [12]:
# nome del file del dataset
file_path = "blood_cell_anomaly_detection.csv"

data_raw = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "alitaqishah/blood-cell-anomaly-detection-2025",
  file_path
)

data_raw.tail()

NameError: name 'KaggleDatasetAdapter' is not defined

Con il metodo info stampiamo le informazioni principali come numero di istanze, data type
per feature, e spazio in memoria

In [ ]:
data_raw.info(memory_usage="deep")

### Significato delle features
Il dataset contiene 36 features, di seguito sono riportate raggruppate per tipologia:

**Morphology** — diametro, circolarità, eccentricità, lobularità, granularità, area del nucleo, densità della cromatina

**Color** — valori RGB medi, intensità della colorazione

**Clinical CBC** — dati ricavati dagli esami del sangue, come quantità globuli bianchi e rossi, piastrine, emoglobina etc.

**Acquisition** — dati riguardanti l'immagine al microscopio, come il modello, la risoluzione e grado di ingrandimento

**AI Scores** — dati legati al modello CytoDiffusion che risolve la stessa tipologia di problema, come confidenza di anomalia della cellula e di suddivisione del tipo della cellula. Troviamo anche il valore di sicurezza della stima del tipo di cellula di un medico esperto

La variabile che tenteremo di predirre è **disease_category**. Le label possono essere:
- __Normal_WBC__, globuli bianchi normali
- __Normal_RBC__, globuli rossi normali
- __Leukemia__
- __Anemia__
- __Sickle_Cell_Anemia__, anemia falciforme
- __Infection__
- __Artefact__


### Scrematura dei dati